# Qualifier-Based MeSH Edge Typing — Step 1

Extracts semantically-typed edges from MeSH annotation qualifiers.  
Replaces generic `co_occurs_with` edges with typed relations  
(`implicated_in`, `associated_with_disorder`, `used_in`, `treated_by`)  
by leveraging the qualifier suffix in MeSH annotations.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Ensure src is on the path when running from experiments/
repo_root = Path("../").resolve()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from neurovlm.gnn.mesh_re import (
    QUALIFIER_TO_RELATION,
    build_qualifier_name_lookup,
    extract_qualifier_edges,
)

DATA = repo_root / "experiments" / "data"

ANNOTATIONS_PATH  = DATA / "mesh_kg"    / "mesh_annotations_long.parquet"
DESCRIPTORS_PATH  = DATA / "mesh_kg"    / "mesh_descriptors.parquet"
KG_NODES_PATH     = DATA / "unified_kg" / "unified_kg_nodes.parquet"
KG_EDGES_PATH     = DATA / "unified_kg" / "unified_kg_edges.parquet"
OUTPUT_PATH       = DATA / "nlp_kg"     / "nlp_kg_edges_qualified.parquet"

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Paths OK")

Paths OK


In [2]:
descriptors_df = pd.read_parquet(DESCRIPTORS_PATH)
print(f"Descriptors shape: {descriptors_df.shape}")
print(descriptors_df.head(5))

name_to_ui = build_qualifier_name_lookup(descriptors_df)
print(f"\nName→UI lookup size: {len(name_to_ui):,}")
# Sample a few entries
sample_keys = list(name_to_ui.keys())[:5]
for k in sample_keys:
    print(f"  '{k}' → {name_to_ui[k]}")

Descriptors shape: (31110, 4)
        ui                    name  \
0  D000001              Calcimycin   
1  D000002                 Temefos   
2  D000003               Abattoirs   
3  D000004  Abbreviations as Topic   
4  D000005                 Abdomen   

                                        tree_numbers  \
0  [D02.355.291.933.125, D02.540.576.625.125, D03...   
1  [D02.705.400.625.800, D02.705.539.345.800, D02...   
2             [J01.576.423.200.700.100, J03.540.020]   
3                          [L01.559.598.400.556.131]   
4                                      [A01.923.047]   

                                            synonyms  
0  [4-Benzoxazolecarboxylic acid, 5-(methylamino)...  
1                           [Temephos, Difos, Abate]  
2  [Abattoir, Slaughterhouses, Slaughterhouse, Sl...  
3                                [Acronyms as Topic]  
4                                         [Abdomens]  

Name→UI lookup size: 31,110
  'calcimycin' → D000001
  'temefos' → D00000

In [3]:
annotations_df = pd.read_parquet(ANNOTATIONS_PATH)
print(f"Annotations shape: {annotations_df.shape}")
print(annotations_df.head(5))

# Qualifier distribution (top 30)
quals = annotations_df["mesh_term"].str.split("/").str[1]
print("\nTop 30 qualifiers in annotations:")
print(quals.value_counts().head(30).to_string())

Annotations shape: (14470827, 2)
       pmid                  mesh_term
0  41747334                     Humans
1  41747334  Ischemic Stroke/diagnosis
2  41747334    Diagnosis, Differential
3  41747334                       Male
4  41747334              Fatal Outcome

Top 30 qualifiers in annotations:
mesh_term
diagnostic imaging             740892
pathology                      723386
methods                        663544
physiology                     572032
diagnosis                      503663
metabolism                     357799
physiopathology                350468
surgery                        323150
complications                  240122
etiology                       228072
chemistry                      171539
genetics                       166683
therapeutic use                130825
therapy                        125937
drug therapy                   119161
drug effects                   105603
adverse effects                 88300
blood                           88271
phar

In [4]:
# Show mapping coverage: which qualifiers map, which are skipped
qual_counts = quals.value_counts()
qual_counts.name = "count"
qual_df = qual_counts.reset_index()
qual_df.columns = ["qualifier", "count"]
qual_df["relation_type"] = qual_df["qualifier"].map(QUALIFIER_TO_RELATION)
qual_df["mapped"] = qual_df["relation_type"].notna()

print("Mapped qualifiers:")
print(qual_df[qual_df["mapped"]].to_string(index=False))

mapped_rows   = qual_df.loc[qual_df["mapped"], "count"].sum()
total_qual    = qual_df["count"].sum()
print(f"\nTotal qualified annotation rows : {total_qual:,}")
print(f"Rows covered by mapping         : {mapped_rows:,}  ({100*mapped_rows/total_qual:.1f}%)")

Mapped qualifiers:
         qualifier  count            relation_type  mapped
diagnostic imaging 740892 associated_with_disorder    True
         pathology 723386            implicated_in    True
           methods 663544                  used_in    True
        physiology 572032            implicated_in    True
         diagnosis 503663 associated_with_disorder    True
        metabolism 357799            implicated_in    True
   physiopathology 350468            implicated_in    True
           surgery 323150               treated_by    True
     complications 240122 associated_with_disorder    True
          etiology 228072 associated_with_disorder    True
          genetics 166683            implicated_in    True
   therapeutic use 130825               treated_by    True
           therapy 125937               treated_by    True
      drug therapy 119161               treated_by    True
      drug effects 105603            implicated_in    True
   adverse effects  88300 associated_

In [5]:
kg_nodes_df = pd.read_parquet(KG_NODES_PATH)
kg_node_ids = set(kg_nodes_df["canonical_id"])
print(f"KG node count: {len(kg_node_ids):,}")

# Run extraction (may take 5–15 minutes)
edges_df = extract_qualifier_edges(
    annotations_df=annotations_df,
    name_to_ui=name_to_ui,
    kg_node_ids=kg_node_ids,
    verbose=True,
)
print(f"\nResult shape: {edges_df.shape}")
print(edges_df.head())

KG node count: 33,784
Total annotation rows :   14,470,827
Rows with qualifier   :    6,860,967  (47.4%)
Terms mapped to UI    :   14,470,827  (100.0%)
Typed term rows       :    5,752,600
Unique (pmid, term)   :   12,169,002
Raw edge pairs        :   72,071,945
After KG node filter  :  144,143,890

Final edge count      :   14,790,106
Elapsed               : 43.4s

Result shape: (14790106, 6)
   subject_id             relation_type object_id          source  weight  \
0  D000083242  associated_with_disorder   D006801  mesh_qualifier  1554.0   
1  D000083242  associated_with_disorder   D003937  mesh_qualifier    27.0   
2  D000083242  associated_with_disorder   D008297  mesh_qualifier  1203.0   
3  D000083242  associated_with_disorder   D017809  mesh_qualifier     1.0   
4  D000083242  associated_with_disorder   D001228  mesh_qualifier     1.0   

        source_kg  
0  mesh_qualifier  
1  mesh_qualifier  
2  mesh_qualifier  
3  mesh_qualifier  
4  mesh_qualifier  


In [6]:
# Edge statistics — count per relation type
print("New edges per relation type:")
new_counts = edges_df["relation_type"].value_counts()
print(new_counts.to_string())

# Compare to current KG
kg_edges_df = pd.read_parquet(KG_EDGES_PATH)
print("\nCurrent unified KG relation distribution:")
current_counts = kg_edges_df["relation_type"].value_counts()
print(current_counts.to_string())

print(f"\nTotal current edges : {len(kg_edges_df):>10,}")
print(f"New qualifier edges : {len(edges_df):>10,}")

New edges per relation type:
relation_type
implicated_in               6619094
associated_with_disorder    4185358
treated_by                  2695376
used_in                     1290278

Current unified KG relation distribution:
relation_type
co_occurs_with              281629
narrower_term_of             42519
associated_with_disorder      2107
implicated_in                 1803
co_activates_with              931
expressed_in                   577

Total current edges :    329,566
New qualifier edges : 14,790,106


In [7]:
# Spot-check: show 5 edges per relation type with human-readable names
ui_to_name = dict(zip(descriptors_df["ui"], descriptors_df["name"]))

# Also include non-MeSH node names from unified KG nodes
extra_names = dict(zip(kg_nodes_df["canonical_id"], kg_nodes_df["name"]))
ui_to_name.update(extra_names)

sample = edges_df.copy()
sample["subject_name"] = sample["subject_id"].map(ui_to_name)
sample["object_name"]  = sample["object_id"].map(ui_to_name)

for rel_type, group in sample.groupby("relation_type"):
    print(f"\n── {rel_type} (showing 5 examples) ──")
    cols = ["subject_name", "relation_type", "object_name", "weight"]
    print(group.nlargest(5, "weight")[cols].to_string(index=False))


── associated_with_disorder (showing 5 examples) ──
              subject_name            relation_type                object_name  weight
                     Brain associated_with_disorder                     Humans 56623.0
                    Humans associated_with_disorder                      Brain 56623.0
                     Brain associated_with_disorder Magnetic Resonance Imaging 40984.0
Magnetic Resonance Imaging associated_with_disorder                      Brain 40984.0
                     Brain associated_with_disorder                       Male 33527.0

── implicated_in (showing 5 examples) ──
subject_name relation_type                object_name   weight
       Brain implicated_in                     Humans 115363.0
      Humans implicated_in                      Brain 115363.0
       Brain implicated_in                       Male  79731.0
        Male implicated_in                      Brain  79731.0
       Brain implicated_in Magnetic Resonance Imaging  74523.0

── t

In [8]:
# Filter verification: confirm both endpoints are in unified_kg_nodes
both_in = (
    edges_df["subject_id"].isin(kg_node_ids)
    & edges_df["object_id"].isin(kg_node_ids)
)
n_both = both_in.sum()
n_total = len(edges_df)
print(f"Edges with both endpoints in unified KG : {n_both:,} / {n_total:,}  ({100*n_both/n_total:.2f}%)")

# Node type breakdown for endpoints
id_to_type = dict(zip(kg_nodes_df["canonical_id"], kg_nodes_df["node_type"]))
edges_df["subject_type"] = edges_df["subject_id"].map(id_to_type)
edges_df["object_type"]  = edges_df["object_id"].map(id_to_type)
print("\nSubject node type distribution:")
print(edges_df["subject_type"].value_counts().to_string())
# Drop temp columns
edges_df = edges_df.drop(columns=["subject_type", "object_type"])

Edges with both endpoints in unified KG : 14,790,106 / 14,790,106  (100.00%)

Subject node type distribution:
subject_type
disorder               4492336
molecular              3067268
method                 2618056
anatomical_region      1768181
biological_process      921930
other                   686744
cognitive_construct     567528
organism                427084
demographic             240979


In [9]:
edges_df.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(edges_df):,} edges to:")
print(f"  {OUTPUT_PATH}")
print(f"\nSchema:")
print(edges_df.dtypes)

Saved 14,790,106 edges to:
  /Users/borng/code/lab_work/neurovlm/experiments/data/nlp_kg/nlp_kg_edges_qualified.parquet

Schema:
subject_id           str
relation_type        str
object_id            str
source               str
weight           float64
source_kg            str
dtype: object


In [10]:
# Final summary: how this changes the unified KG balance if merged
combined_counts = (
    pd.concat(
        [kg_edges_df[["relation_type"]], edges_df[["relation_type"]]],
        ignore_index=True,
    )["relation_type"]
    .value_counts()
)
combined_total = combined_counts.sum()

print("Projected unified KG distribution after merge:")
print(f"{'relation_type':<30}  {'count':>10}  {'%':>7}")
print("-" * 52)
for rel, cnt in combined_counts.items():
    print(f"{rel:<30}  {cnt:>10,}  {100*cnt/combined_total:>6.1f}%")
print("-" * 52)
print(f"{'TOTAL':<30}  {combined_total:>10,}  {100.0:>6.1f}%")

current_cow = current_counts.get("co_occurs_with", 0)
combined_cow = combined_counts.get("co_occurs_with", 0)
print(f"\nco_occurs_with share: {100*current_cow/current_counts.sum():.1f}% → {100*combined_cow/combined_total:.1f}%")

Projected unified KG distribution after merge:
relation_type                        count        %
----------------------------------------------------
implicated_in                    6,620,897    43.8%
associated_with_disorder         4,187,465    27.7%
treated_by                       2,695,376    17.8%
used_in                          1,290,278     8.5%
co_occurs_with                     281,629     1.9%
narrower_term_of                    42,519     0.3%
co_activates_with                      931     0.0%
expressed_in                           577     0.0%
----------------------------------------------------
TOTAL                           15,119,672   100.0%

co_occurs_with share: 85.5% → 1.9%
